[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        return

    def forward(self, x_q, x_kv):

        query_proj = self.W_q(x_q) # 2 X 6 X 64
        key_proj = self.W_k(x_kv) # 2 X 10 X 64
        value_proj = self.W_v(x_kv) # 2 X 10 X 64

        query_proj = query_proj.reshape(query_proj.shape[0], query_proj.shape[1], self.num_heads, self.head_dim) # 2 X 6 X 4 X 16
        key_proj = key_proj.reshape(key_proj.shape[0], key_proj.shape[1], self.num_heads, self.head_dim) # 2 X 10 X 4 X 16
        value_proj = value_proj.reshape(value_proj.shape[0], value_proj.shape[1], self.num_heads, self.head_dim) # 2 X 10 X 4 X 16

        attn_scores = torch.einsum('bqhd, bkhd -> bhqk', query_proj, key_proj) # 2 X 4 X 6 X 10
        attn_scores = attn_scores / math.sqrt(self.head_dim) # 2 X 4 X 6 X 10
        attn_probs = torch.softmax(attn_scores, dim = -1) # softmax calculating over all keys for a query in each head, 2 x 4 x 6 x 10
        out = torch.einsum('bhqk, bkhd -> bhqkd', attn_probs, value_proj) # multiplying each attn score to value projection vector for each head , 2 X 4 X 6 X 10 X 16
        out = out.sum(dim = 3) # 2 x 4 x 6 x 16
        out = out.transpose(1,2) # 2 x 6 x 4 x 16
        out = out.reshape(out.shape[0], out.shape[1], self.d_model) # 2 x 6 x 64
        out = self.W_o(out) # 2 x 6 x 64

        return out

In [4]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.3ms)
  ✅ [2/4] Q and KV different lengths (1.1ms)
  ✅ [3/4] No causal mask — all KV affects all Q (35.0ms)
  ✅ [4/4] Gradient flow (19.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (57.9ms total)
  Progress saved. Run status() to see your dashboard.

